# 03_robust_dedup — 값 퍼짐(신뢰도) 기반 중복정리

**한 줄 요약:** 같은 물질의 IC50 여러 개를 **로그(pIC50)** 로 바꿔 모으고, 값이 너무 튀는(예: 100배 이상 차이) 물질은 **폐기 권장**으로 표시한 시트 `robust_dedup`를 만든다.
**용어:** pIC50 = 9 − log10(IC50[nM]) — 값이 클수록 강함. 로그라서 '배수 차이'를 다루기 쉬움.
**큰 흐름:** ① 읽기·pIC50 계산·집계함수 → ② 물질별 집계 → ③ 정렬 → ④ 요약 → ⑤ 저장 → ⑥ 색칠

> **📌 이 노트북 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]** 순서.
> ③은 그 셀에 **처음 나온** 함수·문법 설명(기초 반복은 *(01에서 설명)* 으로 생략). `# ...`=주석.

### 준비 — 폴더 위치 맞추기
노트북을 어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동한다(그래야 `data/...` 경로가 맞음).

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 **코드 뜯어보기** *(01에서 설명)*: `os.chdir('..')`=상위 폴더로 이동, `os.path.isdir`=폴더 존재 확인, `print`=화면 출력.

### 셀 1 — 준비 + 데이터 읽기
루트 이동, 라이브러리 로드, all_data 시트를 읽어 유효 행만 남긴다.

In [ ]:
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

XL = "data/HSD17B13_IC50_merged.xlsx"

df = pd.read_excel(XL, sheet_name="all_data").dropna(subset=["canonical_smiles", "ic50_nM"])

🔎 **코드 뜯어보기 (셀 1)** *(import/read_excel/dropna는 01·02에서 설명)*
- `from openpyxl.styles import PatternFill` : 엑셀 셀 색칠용 도구.

### 셀 2 — 정확값 표시·pIC50 계산 + 물질별 집계 함수 → 집계 실행
부등호(`>`,`<`)가 아닌 '정확값'만 표시하고, IC50를 pIC50(로그)로 바꾸고, 물질별로 신뢰도를 매기는 함수를 정의해 적용한다.

In [ ]:
rel = df["relation"].astype(str).str.strip()
df["is_exact"] = ~rel.isin([">", "<", ">=", "<="])
df["pIC50"] = 9 - np.log10(df["ic50_nM"].clip(lower=1e-6))


def agg(group):
    ex = group[group["is_exact"]]
    n_exact = len(ex)
    n_qual = len(group) - n_exact
    out = {
        "compound_id": group["compound_id"].dropna().iloc[0] if group["compound_id"].notna().any() else "",
        "smiles": group["smiles"].dropna().iloc[0] if group["smiles"].notna().any() else "",
        "n_exact": n_exact, "n_qualified": n_qual,
        "sources": ", ".join(sorted(group["source"].dropna().unique())),
    }
    if n_exact == 0:
        out.update(ic50_median_nM=np.nan, pIC50_median=np.nan,
                   fold_change=np.nan, range_log=np.nan,
                   confidence="qualified_only", recommend_keep=False)
        return pd.Series(out)
    p = ex["pIC50"]
    med_p = float(p.median())
    rng = float(p.max() - p.min())          # 로그 범위 = log10(max/min)
    fold = 10 ** rng                         # 배수 (max/min)
    if n_exact == 1:
        conf = "single"
    elif rng <= 0.5:
        conf = "high (<=3x)"
    elif rng <= 1.0:
        conf = "good (<=10x)"
    elif rng <= 2.0:
        conf = "moderate (10-100x)"
    else:
        conf = "CONFLICT (>100x)"
    out.update(
        ic50_median_nM=round(10 ** (9 - med_p), 3),
        pIC50_median=round(med_p, 3),
        fold_change=round(fold, 1),
        range_log=round(rng, 2),
        confidence=conf,
        recommend_keep=(conf != "CONFLICT (>100x)"),
    )
    return pd.Series(out)


res = df.groupby("canonical_smiles").apply(agg, include_groups=False).reset_index()
res = res[["canonical_smiles", "compound_id", "smiles", "ic50_median_nM",
           "pIC50_median", "n_exact", "n_qualified", "fold_change", "range_log",
           "confidence", "recommend_keep", "sources"]]

🔎 **코드 뜯어보기 (셀 2)**
- `~rel.isin([">","<",">=","<="])` : `~`=부정. 부등호가 **아닌**(=정확한 `=` 측정) 행을 True로 → `is_exact`.
- `9 - np.log10(df["ic50_nM"].clip(lower=1e-6))` : **pIC50** 계산. `np.log10`=상용로그. `.clip(lower=1e-6)`=아주 작은 값 이상으로 잘라 log 오류 방지.
- `def agg(group):` : 한 물질의 여러 측정(group)을 받아 대표값·신뢰도를 계산하는 함수.
- `if n_exact == 0: out.update(...); return pd.Series(out)` : 정확값이 없으면 별도 처리. `pd.Series(dict)`=딕셔너리를 한 줄 표로.
- `rng = p.max() - p.min()` : 로그 범위(=log10(최대/최소)). `10 ** rng`=배수. `if/elif/else`로 범위에 따라 신뢰도 등급(high/good/moderate/CONFLICT) 부여.
- `df.groupby("...").apply(agg, include_groups=False)` : 각 물질 그룹에 **함수 agg를 통째로 적용**해 한 줄씩 결과를 모음.

### 셀 3 — 신뢰도 낮은(튀는) 물질을 위로 정렬
값이 100배 이상 충돌하는 물질이 맨 위에 오도록 정렬한다.

In [ ]:
order = {"CONFLICT (>100x)": 0, "moderate (10-100x)": 1, "good (<=10x)": 2,
         "high (<=3x)": 3, "single": 4, "qualified_only": 5}
res["_o"] = res["confidence"].map(order).fillna(9)
res = res.sort_values(["_o", "fold_change"], ascending=[True, False]).drop(columns="_o").reset_index(drop=True)

🔎 **코드 뜯어보기 (셀 3)**
- `res["confidence"].map(order)` : 등급 글자를 정렬용 숫자로 **치환**(작을수록 위). `.fillna(9)`=목록에 없으면 9.
- `sort_values(["_o","fold_change"], ascending=[True, False])` : 등급 오름차 → 배수 내림차 순 정렬. `.drop(columns="_o")`=임시 열 제거.

### 셀 4 — 결과 요약 출력
신뢰도 등급별 개수와, 폐기 권장(>100배) 예시를 화면에 먼저 보여준다.

In [ ]:
print("== robust_dedup 계산 결과 ==")
print("물질 수:", len(res))
print(res["confidence"].value_counts().to_string())
print("\n폐기 권장(>100x 충돌) 예시:")
for _, r in res[res["confidence"] == "CONFLICT (>100x)"].head(5).iterrows():
    print(f"  {r['compound_id']}: {r['fold_change']}배 차이, 측정 {r['n_exact']}건")

🔎 **코드 뜯어보기 (셀 4)**
- `.value_counts().to_string()` : 등급별 개수를 세어 보기 좋게. `for _, r in res[...].head(5).iterrows():` : 표의 각 행을 (번호, 행 r)로 반복. `r['fold_change']`=그 행의 값.

### 셀 5 — 엑셀에 저장 (열려 있으면 새 파일로)
결과 시트를 엑셀에 추가한다. 원본이 열려 잠겨 있으면 새 파일로 저장.

In [ ]:
target = XL
try:
    with pd.ExcelWriter(XL, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
        res.to_excel(w, sheet_name="robust_dedup", index=False)
except PermissionError:
    target = XL.replace(".xlsx", "_robust.xlsx")
    print(f"\n[경고] 원본이 열려 있어 저장 불가 → 새 파일로 저장: {target}")
    with pd.ExcelWriter(target, engine="openpyxl") as w:
        res.to_excel(w, sheet_name="robust_dedup", index=False)

🔎 **코드 뜯어보기 (셀 5)**
- `try: ... except PermissionError: ...` : 저장을 시도하고, 파일이 **열려 잠겨** 저장 실패(PermissionError)하면 → 새 파일 이름으로 저장.
- `XL.replace(".xlsx", "_robust.xlsx")` : 파일명 일부를 바꿔 새 경로 만들기.

### 셀 6 — 폐기 권장 행 빨간색 강조
충돌(>100배) 물질 행을 빨간색으로 칠해 눈에 띄게 한다.

In [ ]:
wb = load_workbook(target)
ws = wb["robust_dedup"]
red = PatternFill(start_color="FF9999", end_color="FF9999", fill_type="solid")
conf_col = list(res.columns).index("confidence") + 1
for row in range(2, ws.max_row + 1):
    if ws.cell(row=row, column=conf_col).value == "CONFLICT (>100x)":
        for c in range(1, len(res.columns) + 1):
            ws.cell(row=row, column=c).fill = red
wb.save(target)
print("\n저장 완료 →", target)

🔎 **코드 뜯어보기 (셀 6)** *(openpyxl 색칠은 01에서 설명)*
- `list(res.columns).index("confidence") + 1` : 'confidence' 열이 몇 번째인지 찾기(+1은 엑셀이 1부터 세서). 그 열이 CONFLICT면 그 행 전체를 빨강으로.